In [1]:
#Nếu chạy trên Google Colab thì cần kết nối với máy chủ trước
from google.colab import drive #type :ignore
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import cv2

import torch
import os
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import datasets,transforms
from torchvision.transforms import ToTensor
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
!pip install torchinfo
from torchinfo import summary
%load_ext tensorboard

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST('../data', train=True, download=True,transform=transforms)
test_dataset = datasets.MNIST('../data', train=False,transform=transforms)
train_loader = torch.utils.data.DataLoader(train_dataset,batch_size = 32)
test_loader = torch.utils.data.DataLoader(test_dataset,batch_size = 32)

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 505kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.51MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.11MB/s]


In [5]:

class Net(nn.Module):
    def __init__(self, dropout = 0, activate = 'relu'):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size = 3, padding = 0, bias = False, dilation = 1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size = 3, padding = 0, bias = False, dilation = 1)
        self.max1  =  nn.MaxPool2d(kernel_size=2)
        self.max2  =  nn.MaxPool2d(kernel_size=2)
        
        if activate == 'sigmoid':
            self.activate = nn.Sigmoid()
        elif activate == 'relu':
            self.activate = nn.ReLU()

        self.drop = nn.Dropout(dropout)
        self.flatten = nn.Flatten() # Dùng để chuyển tensor 4D sang 2D
        self.fc1 = nn.Linear(64 * 5 * 5, 512)
        self.fc2 = nn.Linear(512,10)

    def forward(self, X):
        X = self.max1(self.activate(self.conv1(X)))
        X = self.max2(self.activate(self.conv2(X)))
        X = self.drop(X)

        X = self.flatten(X)
        X = self.activate(self.fc1(X))
        X = self.drop(self.fc2(X))
        return X

In [6]:
epochs = 10
criterion = nn.CrossEntropyLoss()
best_val_loss = 1000000.0

def train_and_test(dropout, learning_rate, activation):
# LOGGING: Tạo tên thư mục dựa trên tham số để dễ so sánh trên biểu đồ
    log_name = f"LR_{learning_rate}_DO_{dropout}_ACT_{activation}"
    writer = SummaryWriter(f"runs/{log_name}")
    global best_val_loss

    model = Net(dropout = dropout, activate = activation).to(device)
    optimizer = optim.Adam(model.parameters() ,lr = learning_rate)
    #training phase
    for epoch in range(epochs):
        for batch_inx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            
            loss = criterion(output, target)        
            loss.backward()
            optimizer.step()

            if batch_inx % 100 == 0 :
                #Loggginf
                step = epoch * len(train_loader) + batch_inx
                writer.add_scalar('Loss/Train_Batch', loss.item(), step)

                print('Train Epoch: {} [{}/{} ({:.0f}%)]\tTrain Loss: {:.6f}'.format(
                    epoch, batch_inx * len(data), len(train_loader.dataset),
                    100. * batch_inx / len(train_loader), loss.item()))
    #testing_phase
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for batch_inx, (data, target) in enumerate(test_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            
            loss = criterion(output, target)        
            test_loss += loss.item()
            pred = output.argmax(keepdim = True, dim = 1)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset) 
    accuracy = 100. * correct / len(test_loader.dataset)

    writer.add_scalar('Test/Loss', test_loss, 0) 
    writer.add_scalar('Test/Accuracy', accuracy, 0)

    if (test_loss < best_val_loss):
        best_val_loss = test_loss
    print("***********    TEST_ACC = {}%    ***********".format(accuracy))
    #Hoàn tác lại giá trị best_val_loss để dùng cho lần sau:
    writer.close() # Đóng writer để đẩy dữ liệu xuống ổ cứng
    return best_val_loss

# Thử nghiệm ảnh hưởng của Dropout Rate 


#Dropout = 0.0

In [ ]:
best_val_loss = train_and_test(dropout=0.0, learning_rate = 0.001, activation = 'relu')

Train Epoch: 0 [0/60000 (0%)]	Train Loss: 2.309013
Train Epoch: 0 [3200/60000 (5%)]	Train Loss: 0.206070
Train Epoch: 0 [6400/60000 (11%)]	Train Loss: 0.080701
Train Epoch: 0 [9600/60000 (16%)]	Train Loss: 0.086151
Train Epoch: 0 [12800/60000 (21%)]	Train Loss: 0.090324
Train Epoch: 0 [16000/60000 (27%)]	Train Loss: 0.039333
Train Epoch: 0 [19200/60000 (32%)]	Train Loss: 0.042392
Train Epoch: 0 [22400/60000 (37%)]	Train Loss: 0.039775
Train Epoch: 0 [25600/60000 (43%)]	Train Loss: 0.076085
Train Epoch: 0 [28800/60000 (48%)]	Train Loss: 0.016382
Train Epoch: 0 [32000/60000 (53%)]	Train Loss: 0.127268
Train Epoch: 0 [35200/60000 (59%)]	Train Loss: 0.057116
Train Epoch: 0 [38400/60000 (64%)]	Train Loss: 0.161669
Train Epoch: 0 [41600/60000 (69%)]	Train Loss: 0.053977
Train Epoch: 0 [44800/60000 (75%)]	Train Loss: 0.019663
Train Epoch: 0 [48000/60000 (80%)]	Train Loss: 0.105152
Train Epoch: 0 [51200/60000 (85%)]	Train Loss: 0.075811
Train Epoch: 0 [54400/60000 (91%)]	Train Loss: 0.002023
T

#Dropout rate = 0.5


In [ ]:
best_val_loss = train_and_test(dropout=0.5, learning_rate = 0.001, activation = 'relu')

Train Epoch: 0 [0/60000 (0%)]	Train Loss: 2.269075
Train Epoch: 0 [3200/60000 (5%)]	Train Loss: 1.107462


# Thử nghiệm hàm kích hoạt 


#Hàm ReLU

In [ ]:
best_val_loss = train_and_test(dropout = 0.5, learning_rate = 0.001, activation = 'relu')

#Hàm Sigmoid

In [ ]:
best_val_loss = train_and_test(dropout = 0.5, learning_rate = 0.001, activation = 'sigmoid')